# 💉 Vaccination Data Analysis

## Professional End-to-End Data Analytics Project

**Tools:** Python • Pandas • NumPy • Matplotlib • Seaborn • SQL • Excel • Power BI

### Project objective
Analyze vaccination coverage, disease cases, incidence rates, vaccine introduction, and immunization schedules to identify trends, dose-level drop-offs, coverage gaps, and exploratory relationships between vaccination coverage and disease incidence.

> **Note:** This notebook is for educational and analytical purposes. Correlation does not establish causation, and the analysis should not be interpreted as medical advice.

## 1. Business / Public-Health Questions

This project addresses the following analytical questions:

1. How does vaccination coverage change over time?
2. How do DTP1 and DTP3 coverage compare?
3. Where is there a drop-off between DTP1 and DTP3?
4. Which countries have relatively lower MCV1 coverage?
5. How do reported disease cases change over time?
6. How does disease incidence vary over time?
7. Is there an exploratory relationship between vaccination coverage and disease incidence?
8. When were selected vaccines first introduced?
9. What patterns appear in vaccination schedules and booster rounds?

## 2. Dataset Overview

Five cleaned datasets are used:

| Dataset | Purpose |
|---|---|
| `coverage.csv` | Vaccination coverage by country, year and antigen |
| `cases.csv` | Reported disease cases |
| `incidence.csv` | Disease incidence rates |
| `introduction.csv` | Vaccine introduction status and year |
| `schedule.csv` | Immunization schedule and rounds |

The datasets are stored in `data/cleaned/` when this notebook is used from the GitHub repository. In Google Colab, upload the same folder structure or change `DATA_DIR` below.

In [ ]:
import os
from pathlib import Path
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

DATA_DIR = Path("data/cleaned")
if not DATA_DIR.exists():
    # Helpful fallback for running the notebook from the project root with the ZIP extracted elsewhere.
    candidates = [Path("cleaned_data"), Path("../cleaned_data")]
    for candidate in candidates:
        if candidate.exists():
            DATA_DIR = candidate
            break

print("Using data directory:", DATA_DIR.resolve())

## 3. Load the Datasets

In [ ]:
coverage = pd.read_csv(DATA_DIR / "coverage.csv")
cases = pd.read_csv(DATA_DIR / "cases.csv")
incidence = pd.read_csv(DATA_DIR / "incidence.csv")
introduction = pd.read_csv(DATA_DIR / "introduction.csv")
schedule = pd.read_csv(DATA_DIR / "schedule.csv")

print("Coverage:", coverage.shape)
print("Cases:", cases.shape)
print("Incidence:", incidence.shape)
print("Introduction:", introduction.shape)
print("Schedule:", schedule.shape)

In [ ]:
datasets = {
    "coverage": coverage,
    "cases": cases,
    "incidence": incidence,
    "introduction": introduction,
    "schedule": schedule,
}

summary = pd.DataFrame({
    "dataset": list(datasets.keys()),
    "rows": [df.shape[0] for df in datasets.values()],
    "columns": [df.shape[1] for df in datasets.values()],
    "missing_cells": [int(df.isna().sum().sum()) for df in datasets.values()],
    "duplicate_rows": [int(df.duplicated().sum()) for df in datasets.values()],
})
summary

## 4. Data Quality Checks

The first stage checks missing values, duplicate records, data types, and year ranges. This is important because the five datasets have different structures and measurement fields.

In [ ]:
for name, df in datasets.items():
    print(f"\n{name.upper()} DATA TYPES")
    display(df.dtypes.to_frame("dtype"))

print("\nMissing values")
display(pd.DataFrame({name: df.isna().sum() for name, df in datasets.items()}).fillna(0).astype(int))

In [ ]:
for name, df in datasets.items():
    if "YEAR" in df.columns:
        years = pd.to_numeric(df["YEAR"], errors="coerce").dropna()
        print(f"{name}: {int(years.min())} to {int(years.max())}")

# Standardize year fields where applicable.
for df in [coverage, cases, incidence, introduction, schedule]:
    if "YEAR" in df.columns:
        df["YEAR"] = pd.to_numeric(df["YEAR"], errors="coerce")
        df["YEAR"] = df["YEAR"].round().astype("Int64")

# Numeric measures.
for df, cols in [
    (coverage, ["TARGET_NUMBER", "DOSES", "COVERAGE"]),
    (cases, ["CASES"]),
    (incidence, ["INCIDENCE_RATE"]),
    (schedule, ["SCHEDULEROUNDS"]),
]:
    for col in cols:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")

print("Standardization complete.")

## 5. Exploratory Data Analysis — Vaccination Coverage

In [ ]:
# Focus on official/WUENIC-style coverage estimates where available.
coverage_analytic = coverage.copy()
wu = coverage_analytic[coverage_analytic["COVERAGE_CATEGORY"].astype(str).str.upper().eq("WUENIC")].copy()

print("WUENIC rows:", len(wu))
print("Antigens available:", wu["ANTIGEN"].nunique())
print("Countries available:", wu["CODE"].nunique())

wu["COVERAGE"] = pd.to_numeric(wu["COVERAGE"], errors="coerce")
wu = wu[(wu["COVERAGE"].between(0, 100)) | wu["COVERAGE"].isna()].copy()
wu.head()

In [ ]:
# Annual average coverage for common vaccine indicators.
trend_antigens = [a for a in ["DTPCV1", "DTPCV3", "MCV1"] if a in wu["ANTIGEN"].unique()]
trend = (wu[wu["ANTIGEN"].isin(trend_antigens)]
         .groupby(["YEAR", "ANTIGEN"], as_index=False)["COVERAGE"]
         .mean())

display(trend.head(15))

plt.figure(figsize=(11, 6))
sns.lineplot(data=trend, x="YEAR", y="COVERAGE", hue="ANTIGEN", marker="o")
plt.title("Average Vaccination Coverage Trend")
plt.xlabel("Year")
plt.ylabel("Average coverage (%)")
plt.tight_layout()
plt.show()

## 6. DTP1 vs DTP3 Coverage and Drop-off

DTP1 and DTP3 are compared to examine whether coverage decreases between the first and third dose. This is an important program-monitoring metric because a gap can indicate incomplete continuation through the vaccination series.

In [ ]:
dtp = wu[wu["ANTIGEN"].isin(["DTPCV1", "DTPCV3"])].copy()
dtp_avg = dtp.groupby(["YEAR", "ANTIGEN"], as_index=False)["COVERAGE"].mean()

dtp_pivot = dtp_avg.pivot(index="YEAR", columns="ANTIGEN", values="COVERAGE")
dtp_pivot["DTP1_to_DTP3_dropoff"] = dtp_pivot.get("DTPCV1") - dtp_pivot.get("DTPCV3")

display(dtp_pivot.tail(15))

plt.figure(figsize=(11, 6))
for col in ["DTPCV1", "DTPCV3"]:
    if col in dtp_pivot.columns:
        plt.plot(dtp_pivot.index, dtp_pivot[col], marker="o", label=col)
plt.title("DTP1 vs DTP3 Average Coverage")
plt.xlabel("Year")
plt.ylabel("Coverage (%)")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
dropoff = dtp_pivot.reset_index()[["YEAR", "DTP1_to_DTP3_dropoff"]].dropna()
if not dropoff.empty:
    plt.figure(figsize=(11, 5))
    sns.barplot(data=dropoff, x="YEAR", y="DTP1_to_DTP3_dropoff")
    plt.xticks(rotation=60)
    plt.title("DTP1-to-DTP3 Coverage Drop-off")
    plt.xlabel("Year")
    plt.ylabel("Coverage-point difference")
    plt.tight_layout()
    plt.show()

display(dropoff.sort_values("DTP1_to_DTP3_dropoff", ascending=False).head(10))

## 7. MCV1 Country Coverage — 2023

In [ ]:
mcv1 = wu[(wu["ANTIGEN"] == "MCV1") & (wu["YEAR"] == 2023)].copy()
mcv1_country = (mcv1.groupby(["CODE", "NAME"], as_index=False)["COVERAGE"].mean()
                 .sort_values("COVERAGE", ascending=True))

display(mcv1_country.head(15))

plt.figure(figsize=(10, 7))
top_low = mcv1_country.head(15).sort_values("COVERAGE", ascending=True)
sns.barplot(data=top_low, x="COVERAGE", y="NAME")
plt.title("Countries with Lower MCV1 Coverage — 2023")
plt.xlabel("Coverage (%)")
plt.ylabel("Country")
plt.tight_layout()
plt.show()

## 8. Reported Disease Cases

In [ ]:
cases["CASES"] = pd.to_numeric(cases["CASES"], errors="coerce")
annual_cases = cases.groupby(["YEAR", "DISEASE_DESCRIPTION"], as_index=False)["CASES"].sum()

display(annual_cases.head(15))

# Top diseases by total reported cases across the available period.
top_diseases = (cases.groupby("DISEASE_DESCRIPTION")["CASES"].sum()
                .sort_values(ascending=False).head(8))
display(top_diseases.to_frame("total_reported_cases"))

In [ ]:
plot_df = annual_cases[annual_cases["DISEASE_DESCRIPTION"].isin(top_diseases.index)]
plt.figure(figsize=(12, 7))
sns.lineplot(data=plot_df, x="YEAR", y="CASES", hue="DISEASE_DESCRIPTION")
plt.title("Annual Reported Disease Cases — Selected Diseases")
plt.xlabel("Year")
plt.ylabel("Reported cases")
plt.tight_layout()
plt.show()

## 9. Disease Incidence Analysis

In [ ]:
incidence["INCIDENCE_RATE"] = pd.to_numeric(incidence["INCIDENCE_RATE"], errors="coerce")
annual_incidence = incidence.groupby(["YEAR", "DISEASE_DESCRIPTION"], as_index=False)["INCIDENCE_RATE"].mean()

display(annual_incidence.head(15))

top_incidence_diseases = (incidence.groupby("DISEASE_DESCRIPTION")["INCIDENCE_RATE"].mean()
                           .sort_values(ascending=False).head(8))
display(top_incidence_diseases.to_frame("mean_incidence_rate"))

## 10. Coverage vs Disease Incidence — Exploratory Correlation

To explore whether vaccination coverage and disease incidence move together, country-year observations are aligned using a common key. Because different vaccines and diseases use different units and denominators, this is treated as an exploratory analysis rather than a causal model.

In [ ]:
# Use MCV1 coverage and measles incidence as a focused example.
mc = wu[(wu["ANTIGEN"] == "MCV1")][["CODE", "NAME", "YEAR", "COVERAGE"]].copy()
mc = mc.rename(columns={"COVERAGE": "MCV1_COVERAGE"})

measles = incidence[incidence["DISEASE"] == "MEASLES"][["CODE", "NAME", "YEAR", "INCIDENCE_RATE"]].copy()
measles = measles.rename(columns={"INCIDENCE_RATE": "MEASLES_INCIDENCE"})

merged = mc.merge(measles, on=["CODE", "NAME", "YEAR"], how="inner").dropna()
print("Matched country-year observations:", len(merged))

display(merged.head())

if len(merged) >= 3:
    corr = merged[["MCV1_COVERAGE", "MEASLES_INCIDENCE"]].corr().iloc[0, 1]
    print(f"Pearson correlation: {corr:.3f}")

    plt.figure(figsize=(9, 6))
    sns.scatterplot(data=merged, x="MCV1_COVERAGE", y="MEASLES_INCIDENCE", alpha=0.5)
    plt.title("MCV1 Coverage vs Measles Incidence")
    plt.xlabel("MCV1 coverage (%)")
    plt.ylabel("Measles incidence rate")
    plt.tight_layout()
    plt.show()

## 11. Vaccine Introduction Analysis

In [ ]:
intro = introduction.copy()
intro["YEAR"] = pd.to_numeric(intro["YEAR"], errors="coerce")

# Treat statuses beginning with Yes as an introduction signal.
intro_yes = intro[intro["INTRO"].astype(str).str.startswith("Yes", na=False)].copy()
first_intro = (intro_yes.groupby(["COUNTRYNAME", "DESCRIPTION"], as_index=False)["YEAR"].min()
               .rename(columns={"YEAR": "FIRST_INTRODUCTION_YEAR"}))

display(first_intro.head(20))

# Example: count first introductions by year.
intro_by_year = first_intro.groupby("FIRST_INTRODUCTION_YEAR").size().reset_index(name="new_country_vaccine_introductions")
display(intro_by_year.sort_values("FIRST_INTRODUCTION_YEAR").head(20))

## 12. Immunization Schedule and Booster Rounds

In [ ]:
schedule["SCHEDULEROUNDS"] = pd.to_numeric(schedule["SCHEDULEROUNDS"], errors="coerce")
schedule_summary = (schedule.groupby(["YEAR", "VACCINE_DESCRIPTION"], as_index=False)["SCHEDULEROUNDS"].max())

display(schedule_summary.head(20))

booster_like = schedule[schedule["SCHEDULEROUNDS"] >= 2].copy()
booster_trend = booster_like.groupby("YEAR").size().reset_index(name="schedule_records_with_2plus_rounds")
display(booster_trend)

## 13. Key Findings

Based on the exploratory analysis above, the main areas of insight are:

- Vaccination coverage can be compared across years, countries, and vaccine indicators.
- DTP1-to-DTP3 differences provide a simple measure of dose-series drop-off.
- MCV1 analysis highlights country-level variation in measles-containing vaccine coverage.
- Disease-case and incidence analysis provides a view of disease burden over time.
- Vaccine introduction data supports analysis of when specific vaccines were introduced in different countries.
- Schedule data supports analysis of dose rounds and booster-related patterns.
- Coverage-incidence correlation is useful for exploration but should not be interpreted as proof of causality.

For formal reporting, the project also contains prepared CSV outputs and figures in the `outputs/` and `figures/` folders.

## 14. Project Deliverables

The GitHub project contains:

- Python analysis script
- Cleaned datasets
- SQL schema and analytical queries
- Excel analysis workbooks
- Visualization figures
- Analysis output CSV files
- Professional PDF report
- Power BI dashboard specification and build guide
- This Jupyter/Colab notebook

### Repository workflow

**Data → Cleaning → EDA → Python Analysis → SQL → Visualization → Reporting → Dashboard Planning**

## 15. Conclusion

This project demonstrates an end-to-end data analytics workflow applied to vaccination and public-health datasets. It combines data preparation, exploratory analysis, statistical exploration, visualization, SQL querying, Excel reporting, and Power BI planning into one portfolio project.

The analysis is designed to support exploratory understanding of vaccination coverage, disease patterns, and immunization-program indicators while clearly separating descriptive findings from causal claims.

## 👨‍💻 Author

**Mousam Shivhare**  
B.Sc. (Hons.) Agriculture | Data Analytics & Business Analytics Enthusiast

- GitHub: https://github.com/mousamshivhare123-ops
- LinkedIn: https://www.linkedin.com/in/mousam-shivhare